# PDF Text Extraction and Processing

This notebook demonstrates how to extract text from PDF documents using image models from Amazon Bedrock (Claude 3.7 Sonnet and Amazon Nova Pro). It processes each page of a PDF as an image and extracts its content using parallel asynchronous calls.

## Importing Libraries

In [95]:
import boto3
import json
import fitz
import base64
import time
import asyncio
from concurrent.futures import ThreadPoolExecutor

## Defining Variables and Clients

In [96]:
executor = ThreadPoolExecutor()
loop = asyncio.get_event_loop()
bedrock = boto3.client("bedrock-runtime")
instructions = """Please extract and format the readable text from the provided image, respecting the original structure as much as possible. Follow these instructions:

- For continuous text, keep the original separation by line breaks.
- If there are tables, use Markdown syntax to present them in an organized way:

Example of expected output:

- For plain text: [Extracted text with line breaks as necessary]

- For documents with tables:
| Header1 | Header2 | Header3 |
|---------|---------|---------|
|  Data1  |  Value1 |  Value1 |
|  Data2  |  Value2 |  Value2 |

Note: Avoid adding additional interpretations or comments to the extracted content."""

## Bedrock Response

In [97]:
def bedrock_response(model_id, request_body, start_time):
    response = bedrock.invoke_model(
        modelId=model_id,
        body=json.dumps(request_body),
    )

    response = json.loads(response["body"].read())
    response["start_time"] = start_time
    response["end_time"] = time.time()
    return response

## Claude 3.7 Sonnet Structure

In [98]:
def extract_text_with_claude_3_7_sonnet(base64_image):
    start_time = time.time()
    model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 2048,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": instructions,
                    },
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": base64_image,
                        },
                    },
                ],
            }
        ]
    }

    return bedrock_response(model_id, request_body, start_time)

## Amazon Nova Pro Structure

In [99]:
def extract_text_with_nova_pro(base64_image):
    start_time = time.time()
    model_id = "us.amazon.nova-pro-v1:0"

    request_body = {
        "schemaVersion": "messages-v1",
        "inferenceConfig": {
            "maxTokens": 2048,
        },
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "text": instructions,
                    },
                    {
                        "image": {"format": "jpeg", "source": {"bytes": base64_image}}
                    },
                ],
            }
        ]
    }

    return bedrock_response(model_id, request_body, start_time)

## Generate Image From PDF and Process Extraction

In [100]:
document = fitz.open("./documents/sample.pdf")

for page_number, page in enumerate(document):
    document_image = f"./images/page_{page_number + 1}.jpeg"
    pix = page.get_pixmap(alpha=False, dpi=300)
    pix.save(document_image)

    with open(document_image, "rb") as image_file:
        document_image = base64.b64encode(image_file.read()).decode("utf8")

    async def parallel_process():
        return await asyncio.gather(
            loop.run_in_executor(executor, extract_text_with_claude_3_7_sonnet, document_image),
            loop.run_in_executor(executor, extract_text_with_nova_pro, document_image)
        )

    claude_3_7_sonnet_result, nova_pro_result = await parallel_process()

    with open(f"./texts/page_{page_number + 1}.txt", "w") as f:
        f.write(f"Claude 3.7 Sonnet: {claude_3_7_sonnet_result["content"][0]["text"]}")
        f.write("\n\n")
        f.write(f"Amazon Nova Pro: {nova_pro_result["output"]["message"]["content"][0]["text"]}")

## Print LLMs Metrics

In [101]:
print(f"""
Claude 3.7 Sonnet:
  Input Tokens  : {claude_3_7_sonnet_result["usage"]["input_tokens"]}
  Output Tokens : {claude_3_7_sonnet_result["usage"]["output_tokens"]}
  Start Time    : {claude_3_7_sonnet_result["start_time"]}
  End Time      : {claude_3_7_sonnet_result["end_time"]}

Amazon Nova Pro:
  Input Tokens  : {nova_pro_result["usage"]["inputTokens"]}
  Output Tokens : {nova_pro_result["usage"]["outputTokens"]}
  Start Time    : {nova_pro_result["start_time"]}
  End Time      : {nova_pro_result["end_time"]}
""")


Claude 3.7 Sonnet:
  Input Tokens  : 1666
  Output Tokens : 1036
  Start Time    : 1746124263.978323
  End Time      : 1746124292.999416

Amazon Nova Pro:
  Input Tokens  : 2223
  Output Tokens : 971
  Start Time    : 1746124263.98382
  End Time      : 1746124279.478841

